# Exercise 11 — Case: Buffett's Alpha

Frazzini, Kabiller & Pedersen (2018), *Buffett's Alpha*, **Financial Analysts Journal**
74(4), 35–55.

This notebook is **question 8 of exercise sheet 11**, discussed on Monday 30 November
2026. Questions 1 to 7 are answered from the paper. Question 8 takes the paper onto
Berkshire Hathaway's own listed returns: it rebuilds the track record and the alpha ladder,
splits the sample where the published study ends, and then asks the Lecture 11 question —
whether either result has enough statistical power to mean anything.

**Data.** Berkshire Hathaway Class A monthly total returns, May 1985 to December 2025,
derived from S&P Capital IQ adjusted closing prices. The Fama–French five factors and
the momentum factor from the [Kenneth R. French Data
Library](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html), and
the betting-against-beta factor from [AQR Capital
Management](https://www.aqr.com/Insights/Datasets) — the same files as Exercise 10.

**Two departures from the paper, both deliberate.**

1. The paper runs October 1976 to March 2017. Capital IQ's price history for BRK.A
   begins in **April 1985**, so we lose Berkshire's strongest decade. Every level in
   this notebook is therefore lower than the published one. The *pattern* is what
   replicates, not the magnitudes.
2. The paper's quality factor is AQR's QMJ. We use Fama–French **RMW** (robust minus
   weak profitability) instead. It is a narrower measure of quality — profitability
   only, where QMJ also carries growth, safety and payout — so it should and does load
   less strongly.

**How to work with this notebook.** Start with **Runtime → Restart and run all**, then
read top to bottom. Section headings match the parts of question 8 on the sheet. Parts (a)
and (b) are computed for you. In the student version three code cells are left for you to
write — the alpha ladder in (c), the power check and the simulation in (d) — each marked
`TODO` with what it must produce. How you write them is up to you, AI included. The figure
below each one runs once it is done.

In [ ]:
!wget -q https://raw.githubusercontent.com/KroeTiA/Investments/main/exercise_utils.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from matplotlib.ticker import PercentFormatter

from exercise_utils import FHNW, setup_style
setup_style()

SEED = 42
rng = np.random.default_rng(SEED)

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")


def as_percent(df, cols):
    # display copy with the named columns scaled to percentage points
    out = df.copy()
    out[cols] = out[cols] * 100
    return out.to_string(float_format=lambda v: f"{v:8.2f}")

### The data

Five files, three formats. The Berkshire returns and balance sheet are ours and arrive
clean. The French files are committed verbatim, copyright lines and all, so the cleaning
happens here in front of you — the same `read_french` helper as Exercise 10. The AQR
workbook has nineteen rows of disclaimer above the header and one column per country.

Everything is monthly and in decimals. The merge is an inner join, so the sample is
whatever all four return files have in common.

In [ ]:
BASE = ("https://raw.githubusercontent.com/KroeTiA/Investments/main/"
        "Exercise_11/data/")


def read_french(url, skiprows):
    # read one block of a Kenneth French CSV into monthly decimal returns
    df = pd.read_csv(url, skiprows=skiprows)
    df = df.rename(columns={df.columns[0]: "date"})
    df["date"] = df["date"].astype(str).str.strip()
    is_month = df["date"].str.fullmatch(r"\d{6}")
    stop = (~is_month).values.argmax() if (~is_month).any() else len(df)
    df = df.iloc[:stop]
    df = df.set_index(pd.PeriodIndex(df["date"], freq="M")).drop(columns="date")
    df.columns = [c.strip() for c in df.columns]
    return df.astype(float).replace([-99.99, -999.0], np.nan) / 100


brk = pd.read_csv(BASE + "brk_monthly_returns.csv", parse_dates=["Date"])
brk = brk.set_index(pd.PeriodIndex(brk["Date"], freq="M"))[["BRK"]]

ff5 = read_french(BASE + "F-F_Research_Data_5_Factors_2x3.csv", 4)
mom = read_french(BASE + "F-F_Momentum_Factor.csv", 13)

bab = pd.read_excel(BASE + "Betting_Against_Beta_Equity_Factors_Monthly.xlsx",
                    sheet_name="BAB Factors", skiprows=18)
bab["date"] = pd.to_datetime(bab["DATE"], format="%m/%d/%Y")
bab = bab.set_index(pd.PeriodIndex(bab["date"], freq="M"))[["USA"]]
bab = bab.rename(columns={"USA": "BAB"}).dropna()

data = brk.join([ff5, mom, bab], how="inner").dropna()
data["rx"] = data["BRK"] - data["RF"]          # Berkshire excess return

FACTORS = ["Mkt-RF", "SMB", "HML", "Mom", "BAB", "RMW"]

# the two sub-periods: the published sample, and everything after it
A = data.loc[:"2017-12"]
B = data.loc["2018-01":]

print(f"merged sample {data.index[0]} to {data.index[-1]}, "
      f"T = {len(data)} months ({len(data)/12:.1f} years)")
print(f"  period A     {A.index[0]} to {A.index[-1]}, T = {len(A)}")
print(f"  period B     {B.index[0]} to {B.index[-1]}, T = {len(B)}")

## Q8 (a) — The track record

Before any model: how good was it, and against what?

The four numbers that matter are the average excess return, the volatility, the ratio of
the two, and the same ratio for the market. The paper reports a Sharpe ratio of 0.79 for
Berkshire against 0.49 for the market, and calls the first "very good but not
superhuman". The cell below reproduces those four numbers on each sub-period. Say what
changed.

In [ ]:
def summary(df, label):
    rx, mkt = df["rx"], df["Mkt-RF"]
    return pd.Series({
        "months": len(df),
        "BRK mean": rx.mean() * 12,
        "BRK vol": rx.std() * np.sqrt(12),
        "BRK Sharpe": rx.mean() / rx.std() * np.sqrt(12),
        "MKT mean": mkt.mean() * 12,
        "MKT vol": mkt.std() * np.sqrt(12),
        "MKT Sharpe": mkt.mean() / mkt.std() * np.sqrt(12),
    }, name=label)


track = pd.DataFrame([summary(data, "full 1985-2025"),
                      summary(A, "A  1985-2017"),
                      summary(B, "B  2018-2025")])

print("means and volatilities in % p.a.\n")
print(as_percent(track, ["BRK mean", "BRK vol", "MKT mean", "MKT vol"]))

In [ ]:
fig1, ax = plt.subplots(figsize=(7.5, 4.5))
for series, name, colour in [(data["BRK"], "Berkshire Hathaway", FHNW["blue"]),
                             (data["Mkt-RF"] + data["RF"], "US market", FHNW["navy"])]:
    ax.plot(data.index.to_timestamp(), (1 + series).cumprod(),
            label=name, color=colour, lw=1.5)
ax.axvline(pd.Timestamp("2018-01-01"), color=FHNW["red"], lw=1.0, ls="--")
ax.annotate("published\nsample ends", xy=(pd.Timestamp("2018-01-01"), 3),
            xytext=(-8, 0), textcoords="offset points",
            color=FHNW["red"], fontsize=9, ha="right", va="center")
ax.set_yscale("log")
ax.set_ylabel("value of 1 USD invested, log scale")
ax.set_xlabel("")
ax.legend(loc="upper left")
plt.show()

## Q8 (b) — Leverage

The paper estimates Berkshire's leverage at about 1.7 to 1 at market values, from

$$L_t = \frac{TA_t - Cash_t}{Equity_t}$$

and then makes a one-line argument: apply 1.7× to the market and you get roughly 12.7 %,
far short of Berkshire's 18.6 %, so leverage cannot be the whole story.

The cell below computes the same ratio on book values and re-runs the paper's one-line
argument on each of our two sub-periods.

In [ ]:
bs = pd.read_csv(BASE + "brk_balance_sheet_annual.csv",
                 parse_dates=["FiscalYearEnd"])
bs["L"] = (bs["TotalAssets"] - bs["Cash"]) / bs["TotalEquity"]

print(f"book leverage, {bs['FiscalYearEnd'].dt.year.min()}"
      f"-{bs['FiscalYearEnd'].dt.year.max()}:  "
      f"mean {bs['L'].mean():.2f}   median {bs['L'].median():.2f}   "
      f"min {bs['L'].min():.2f}   max {bs['L'].max():.2f}")

L_PAPER = 1.7
print("\nthe paper's one-line argument, re-run:\n")
for df, label in [(data, "full 1985-2025"), (A, "A  1985-2017"), (B, "B  2018-2025")]:
    lev_mkt = L_PAPER * df["Mkt-RF"].mean() * 12
    print(f"  {label:16s}  1.7 x market = {lev_mkt:6.1%}   "
          f"Berkshire = {df['rx'].mean() * 12:6.1%}   "
          f"{'market wins' if lev_mkt > df['rx'].mean() * 12 else 'Berkshire wins'}")

## Q8 (c) — The alpha ladder

This is Table 4 of the paper. Regress Berkshire's excess return on a widening set of
factors and watch the intercept:

$$r_t - r^f_t = \alpha + \beta_1 MKT_t + \beta_2 SMB_t + \beta_3 HML_t
+ \beta_4 UMD_t + \beta_5 BAB_t + \beta_6 RMW_t + \varepsilon_t$$

Five rungs: the market alone, then the three Fama–French factors, then momentum, then
betting-against-beta, then profitability. Run the ladder separately on each sub-period.

The question is not whether the alpha falls — adding regressors can only reduce the
unexplained part. The question is *which* rung takes it, and whether what remains is
still distinguishable from zero.

In [ ]:
LADDER = [("CAPM", ["Mkt-RF"]),
          ("+ FF3", ["Mkt-RF", "SMB", "HML"]),
          ("+ UMD", ["Mkt-RF", "SMB", "HML", "Mom"]),
          ("+ BAB", ["Mkt-RF", "SMB", "HML", "Mom", "BAB"]),
          ("+ RMW", ["Mkt-RF", "SMB", "HML", "Mom", "BAB", "RMW"])]

In [ ]:
# TODO: write ladder(df, label). For each rung in LADDER, regress df["rx"] on a constant
# TODO: and the rung's factors (sm.OLS). Keep the alpha annualised (x 12), its t-statistic,
# TODO: the factor loadings and the R2. Return a DataFrame with one row per rung and the
# TODO: columns "alpha", "t(alpha)", the six factor names and "R2", and print it.
# TODO: The figure below reads the columns "alpha" and "t(alpha)".
def ladder(df, label):
    ...
lad_A = ladder(A, "Period A — May 1985 to December 2017")
lad_B = ladder(B, "Period B — January 2018 to December 2025")

In [ ]:
fig2, ax = plt.subplots(figsize=(7.5, 4.5))
x = np.arange(len(LADDER))
width = 0.38
for offset, tab, name, colour in [(-width / 2, lad_A, "A  1985-2017", FHNW["blue"]),
                                  (+width / 2, lad_B, "B  2018-2025", FHNW["orange"])]:
    bars = ax.bar(x + offset, tab["alpha"], width, label=name, color=colour)
    for rect, t in zip(bars, tab["t(alpha)"]):
        ax.annotate(f"t={t:.2f}", (rect.get_x() + rect.get_width() / 2,
                                   rect.get_height()),
                    textcoords="offset points",
                    xytext=(0, 3 if rect.get_height() >= 0 else -12),
                    ha="center", fontsize=8)
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x, [n for n, _ in LADDER])
ax.set_ylim(-0.022, 0.105)
ax.set_ylabel("alpha, % p.a.")
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.legend()
plt.show()

## Q8 (d) — Does either result have enough power?

Lecture 11 gave the rule

$$t(\alpha) \;\approx\; IR \cdot \sqrt{T}$$

with $T$ in years. Check it against both sub-periods, then use it in the other
direction: how long a track record would each information ratio need before its
t-statistic reached 2?

Then the question that decides how Period B should be read. **Suppose Berkshire's
Period-A skill continued into Period B, completely unchanged.** How often would eight
years of data detect it?

In [ ]:
# TODO: for each period, run the CAPM regression of rx on Mkt-RF. The information ratio
# TODO: is the alpha divided by the residual standard deviation (both monthly), annualised
# TODO: with sqrt(12). Show, per period: the annualised IR, T in years, IR * sqrt(T), the
# TODO: regression's t(alpha), and the years an IR this size needs to reach t = 2.
for df, label in [(A, "A  1985-2017"), (B, "B  2018-2025")]:
    ...

In [ ]:
# how often would eight years detect Period A's skill, if it were still there?
# TODO: take Period A's CAPM alpha and residual standard deviation (monthly). Draw n_sims
# TODO: track records of n_months each from a normal distribution with that mean and
# TODO: standard deviation, using rng. Compute each record's t-statistic of the mean and
# TODO: call the array t_sim. Report the share with t > 1.96, the median t and the share
# TODO: with a negative mean, and Period B's observed CAPM t-statistic, called t_B.
# TODO: The figure below uses t_sim and t_B.
n_months, n_sims = len(B), 20_000

In [ ]:
fig3, ax = plt.subplots(figsize=(7.5, 4.5))
counts, edges, _ = ax.hist(t_sim, bins=60, color=FHNW["blue"], alpha=0.55,
                           edgecolor="none")
mids = (edges[:-1] + edges[1:]) / 2
ax.bar(mids[mids > 1.96], counts[mids > 1.96], width=np.diff(edges)[0],
       color=FHNW["green"], label=f"detected: {(t_sim > 1.96).mean():.0%}")
ax.axvline(1.96, color=FHNW["navy"], lw=1.4, ls="--", label="t = 1.96")
ax.axvline(t_B, color=FHNW["red"], lw=1.8,
           label=f"observed in period B, t = {t_B:.2f}")
ax.set_ylim(0, counts.max() * 1.35)
ax.set_xlabel("t-statistic of the CAPM alpha over an eight-year window")
ax.set_ylabel("simulated track records")
ax.legend(loc="upper right")
plt.show()

### Back to the sheet

Take your two ladders and your power table back to question 7, the investment committee,
and check whether they change your decision. Bring your written answers to class on
Monday 30 November. Drill 11 is in Moodle and closes on Sunday at 23:59.